In [1]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
def step_env(state, action):
    # Aleatoric noise = randomness in the environment
    noise = np.random.normal(0, 0.05)

    next_state = state + action + noise

    # State closer to 0 gives a better reward
    reward = -(next_state ** 2)

    return next_state, reward

In [3]:
class ProbNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, s, a):

        x = torch.cat([s, a], dim=-1)

        out = self.net(x)

        mean = out[:, 0:1]
        logvar = out[:, 1:2]

        return mean, logvar

    def loss(self, s, a, s_next):

        mean, logvar = self.forward(s, a)

        var = torch.exp(logvar)

        # Gaussian Negative Log Likelihood
        loss = 0.5 * ((s_next - mean) ** 2 / var + logvar)

        return loss.mean()

In [4]:
N_MODELS = 3

models = [ProbNet() for _ in range(N_MODELS)]

optims = [
    torch.optim.Adam(model.parameters(), lr=1e-2)
    for model in models
]

print("Number of models:", len(models))

Number of models: 3


In [5]:
def train_ensemble(S, A, S2, epochs=100):

    n = len(S)

    for model, optimizer in zip(models, optims):

        # Bootstrap sampling
        idx = np.random.randint(0, n, n)

        s = S[idx]
        a = A[idx]
        s2 = S2[idx]

        # Convert NumPy arrays to tensors
        s_t = torch.tensor(
            s, dtype=torch.float32
        ).unsqueeze(1)

        a_t = torch.tensor(
            a, dtype=torch.float32
        ).unsqueeze(1)

        s2_t = torch.tensor(
            s2, dtype=torch.float32
        ).unsqueeze(1)

        for _ in range(epochs):

            optimizer.zero_grad()

            loss = model.loss(
                s_t,
                a_t,
                s2_t
            )

            loss.backward()

            optimizer.step()

In [11]:
def imagine_next_state(state, action):

    # Randomly choose one model from ensemble
    model = models[np.random.randint(N_MODELS)]

    s_t = torch.tensor(
        [[state]],
        dtype=torch.float32
    )

    a_t = torch.tensor(
        [[action]],
        dtype=torch.float32
    )

    with torch.no_grad():

        mean, logvar = model(s_t, a_t)

        std = torch.exp(0.5 * logvar)

        sample = mean + std * torch.randn_like(mean)

    return sample.item()

In [13]:
def cem_plan(
    state,
    horizon=5,
    n_candidates=100,
    n_elites=10,
    n_iters=3
):

    mean = np.zeros(horizon)
    std = np.ones(horizon)

    for _ in range(n_iters):

        # Generate candidate action sequences
        candidates = np.random.normal(
            mean,
            std,
            size=(n_candidates, horizon)
        )

        rewards = np.zeros(n_candidates)

        for c in range(n_candidates):

            s = state
            total_reward = 0

            for t in range(horizon):

                s = imagine_next_state(
                    s,
                    candidates[c, t]
                )

                total_reward += -(s ** 2)

            rewards[c] = total_reward

        # Select best candidates
        elite_idx = rewards.argsort()[-n_elites:]

        elites = candidates[elite_idx]

        # Update distribution
        mean = elites.mean(axis=0)

        std = elites.std(axis=0) + 1e-3

    # Return first action
    return mean[0]

In [15]:
state = np.random.uniform(-5, 5)

S = []
A = []
S2 = []

for _ in range(200):

    action = np.random.uniform(-1, 1)

    next_state, reward = step_env(
        state,
        action
    )

    S.append(state)
    A.append(action)
    S2.append(next_state)

    state = next_state

# Convert lists into NumPy arrays
S = np.array(S)
A = np.array(A)
S2 = np.array(S2)

print("Training samples:", len(S))

Training samples: 200


In [17]:
train_ensemble(S, A, S2)

print("Initial model training completed!")

Initial model training completed!


In [20]:
state = 5.0

for t in range(15):

    # Find best action using CEM
    action = cem_plan(state)

    # Perform action in environment
    next_state, reward = step_env(
        state,
        action
    )

    # Add new experience to dataset
    S = np.append(S, state)
    A = np.append(A, action)
    S2 = np.append(S2, next_state)

    print(
        f"t={t:2d} "
        f"state={state:6.2f} "
        f"action={action:6.2f} "
        f"reward={reward:6.2f}"
    )

    state = next_state

    # Retrain models with new data
    train_ensemble(S, A, S2)

t= 0 state=  5.00 action= -2.94 reward= -4.10
t= 1 state=  2.03 action= -2.00 reward= -0.01
t= 2 state=  0.08 action= -0.13 reward= -0.01
t= 3 state= -0.09 action=  0.07 reward= -0.02
t= 4 state= -0.13 action=  0.06 reward= -0.03
t= 5 state= -0.16 action=  0.19 reward= -0.01
t= 6 state=  0.09 action= -0.07 reward= -0.01
t= 7 state=  0.12 action=  0.03 reward= -0.05
t= 8 state=  0.22 action= -0.12 reward= -0.02
t= 9 state=  0.15 action= -0.19 reward= -0.00
t=10 state= -0.01 action=  0.03 reward= -0.00
t=11 state=  0.02 action=  0.04 reward= -0.01
t=12 state=  0.09 action= -0.09 reward= -0.00
t=13 state=  0.00 action= -0.08 reward= -0.01
t=14 state= -0.10 action=  0.16 reward= -0.01
